# Seminar 11: Item2Item Service

Item2Item service is a candidate-generation component. It takes positive items from the user history, retrieves nearest items from several embedding spaces, filters irrelevant pairs, and ranks the remaining candidates by attractiveness.


## 0. Offline Datasets

Before the seminar we construct only raw supervision datasets. No features are stored in these files.

Relevance dataset columns:

- `leftId`
- `rightId`
- `verdict`: `1` for relevant, `0` for irrelevant

Attractiveness dataset columns:

- `anchorId`: positive source item `A`
- `winnerId`: item `B` with higher watch time
- `loserId`: item `C` with lower watch time
- `target`: `log1p(timespent_B) - log1p(timespent_C)`

All features are calculated and joined in this notebook.


In [ ]:
# Run before the seminar from seminar_11_i2i_service/:
#
# python scripts/build_i2i_datasets.py \
#   --data-dir VK-LSVD \
#   --output-dir prepared \
#   --weeks 0 1 2 3 \
#   --max-users 50000 \
#   --max-items 100000 \
#   --relevance-positive-pairs 100000 \
#   --relevance-negative-pairs 100000 \
#   --attractiveness-triplets 200000


## 1. Imports and Paths


In [22]:
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import scipy.sparse as sp
from catboost import CatBoostClassifier, CatBoostRegressor
from implicit.als import AlternatingLeastSquares
from sklearn.metrics import roc_auc_score, average_precision_score, mean_absolute_error
from sklearn.neighbors import NearestNeighbors

pd.options.display.max_columns = 100


In [24]:
SEMINAR_DIR = Path.cwd()
if not (SEMINAR_DIR / "scripts" / "build_i2i_datasets.py").exists():
    SEMINAR_DIR = SEMINAR_DIR / "seminar_11_i2i_service"

DATA_DIR = SEMINAR_DIR / "VK-LSVD"
DATASETS_DIR = SEMINAR_DIR / "prepared"
SUBSAMPLE_NAME = "up0.001_ip0.001"
WEEKS = [0, 1, 2, 3]
CONTENT_DIM = 64
ALS_FACTORS = 32
ALS_ITERATIONS = 10
# watch_ratio = view_time_sec / len_video_sec
POSITIVE_WATCH_RATIO = 0.8

if not DATASETS_DIR.exists():
    raise FileNotFoundError(
        f"Raw datasets were not found at {DATASETS_DIR}. "
        "Run scripts/build_i2i_datasets.py before the seminar."
    )


## 2. Load Raw Supervision Datasets


In [25]:
relevance_train = pd.read_parquet(DATASETS_DIR / "relevance_train.parquet")
relevance_test = pd.read_parquet(DATASETS_DIR / "relevance_test.parquet")
attractiveness_train = pd.read_parquet(DATASETS_DIR / "attractiveness_train.parquet")
attractiveness_test = pd.read_parquet(DATASETS_DIR / "attractiveness_test.parquet")

print(relevance_train.shape, relevance_test.shape)
print(attractiveness_train.shape, attractiveness_test.shape)
relevance_train.head()

(160000, 3) (40000, 3)
(7969, 4) (1993, 4)


,leftId,rightId,verdict
0,175970818,289549133,1
1,49413784,383102533,1
2,220082115,39180312,0
3,199030149,565137913,0
4,66676269,440528741,1


In [26]:
attractiveness_train.head()

,anchorId,winnerId,loserId,target
0,154887242,308804716,492866071,4.043051
1,76978620,390986725,327484273,4.852030
2,586164105,329362081,517932985,4.852030
3,424834082,278659326,45421954,4.852030
4,408285968,316241258,284345673,4.852030


## 3. Load Interaction Context

The raw datasets contain only labels. We load interactions and metadata here to calculate positives, item counters, content embeddings, and collaborative embeddings.


In [27]:
def read_interactions(weeks):
    columns = [
        "user_id",
        "item_id",
        "timespent",
        "like",
        "dislike",
        "share",
        "bookmark",
        "open_comments",
    ]
    frames = []
    for week in weeks:
        path = DATA_DIR / f"subsamples/{SUBSAMPLE_NAME}/train/week_{week:02}.parquet"
        frame = pd.read_parquet(path, columns=columns)
        frame["week"] = week
        frames.append(frame)
    return pd.concat(frames, ignore_index=True)

interactions = read_interactions(WEEKS)
needed_items = pd.unique(pd.concat([
    relevance_train["leftId"], relevance_train["rightId"],
    relevance_test["leftId"], relevance_test["rightId"],
    attractiveness_train["anchorId"], attractiveness_train["winnerId"], attractiveness_train["loserId"],
    attractiveness_test["anchorId"], attractiveness_test["winnerId"], attractiveness_test["loserId"],
], ignore_index=True)).astype(np.uint32)

interactions = interactions[interactions["item_id"].isin(needed_items)].copy()
print("interactions for seminar items:", len(interactions))

interactions for seminar items: 3962461


In [28]:
items_metadata = pd.read_parquet(
    DATA_DIR / "metadata/items_metadata.parquet",
    columns=["item_id", "author_id", "duration", "train_interactions_rank"],
)
items_metadata = items_metadata[items_metadata["item_id"].isin(needed_items)].copy()
interactions = interactions.merge(items_metadata[["item_id", "duration"]], on="item_id", how="inner")

watch_ratio = interactions["timespent"] / interactions["duration"].replace(0, np.nan)
interactions["is_positive"] = (
    interactions["like"]
    | interactions["share"]
    | interactions["bookmark"]
    | interactions["open_comments"]
    | (watch_ratio >= POSITIVE_WATCH_RATIO)
) & ~interactions["dislike"]

user_positives = interactions[interactions["is_positive"]].copy()
print("positive events:", len(user_positives))


positive events: 1043855


## 4. Calculate Item Features

Feature families:

- content similarity from VK-LSVD content embeddings;
- collaborative similarity from ALS item factors;
- item counters and metadata.


In [29]:
def normalize_rows(matrix, eps=1e-12):
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    return matrix / np.maximum(norms, eps)

item_counters = (
    interactions.groupby("item_id", as_index=False)
    .agg(
        shows=("item_id", "size"),
        likes=("like", "sum"),
        dislikes=("dislike", "sum"),
        shares=("share", "sum"),
        bookmarks=("bookmark", "sum"),
        open_comments=("open_comments", "sum"),
        mean_timespent=("timespent", "mean"),
    )
)
item_counters["like_rate"] = item_counters["likes"] / item_counters["shows"]
item_counters["share_rate"] = item_counters["shares"] / item_counters["shows"]
item_counters["bookmark_rate"] = item_counters["bookmarks"] / item_counters["shows"]
item_counters["comment_rate"] = item_counters["open_comments"] / item_counters["shows"]

item_features = (
    items_metadata.merge(item_counters, on="item_id", how="left")
    .fillna(0)
    .sort_values("item_id")
    .reset_index(drop=True)
)
item_ids = item_features["item_id"].to_numpy(dtype=np.uint32)
item_to_idx = {int(item_id): idx for idx, item_id in enumerate(item_ids.tolist())}
item_features.head()


,item_id,author_id,duration,train_interactions_rank,shows,likes,dislikes,shares,bookmarks,open_comments,mean_timespent,like_rate,share_rate,bookmark_rate,comment_rate
0,66761,451170,92,11105,98,0,0,0,0,2,38.887755,0.000000,0.000000,0.000000,0.020408
1,187690,1195903,51,8255,2446,8,1,0,0,33,12.448078,0.003271,0.000000,0.000000,0.013491
2,425947,376241,56,3151,18,1,0,0,0,0,25.500000,0.055556,0.000000,0.000000,0.000000
3,566082,525496,5,6237,660,24,0,3,1,2,4.760606,0.036364,0.004545,0.001515,0.003030
4,618086,392231,40,14639,38,0,0,0,0,0,36.210526,0.000000,0.000000,0.000000,0.000000


In [30]:
with np.load(DATA_DIR / "metadata/item_embeddings.npz") as data:
    all_item_ids = data["item_id"]
    mask = np.isin(all_item_ids, item_ids)
    content_item_ids = all_item_ids[mask].astype(np.uint32)
    content_embeddings = data["embedding"][mask, :CONTENT_DIM].astype(np.float32)

order = np.argsort(content_item_ids)
content_item_ids = content_item_ids[order]
content_embeddings = normalize_rows(content_embeddings[order])

# Keep only items that have content embeddings.
item_features = item_features[item_features["item_id"].isin(content_item_ids)].sort_values("item_id").reset_index(drop=True)
item_ids = item_features["item_id"].to_numpy(dtype=np.uint32)
item_to_idx = {int(item_id): idx for idx, item_id in enumerate(item_ids.tolist())}
content_embeddings = content_embeddings[np.isin(content_item_ids, item_ids)]
interactions = interactions[interactions["item_id"].isin(item_ids)].copy()
user_positives = user_positives[user_positives["item_id"].isin(item_ids)].copy()

print(content_embeddings.shape)


(6240, 64)


In [31]:
def train_als_item_embeddings(interactions, item_ids):
    user_codes, _ = pd.factorize(interactions["user_id"], sort=True)
    item_codes = interactions["item_id"].map(item_to_idx).to_numpy()
    confidence = np.where(interactions["is_positive"].to_numpy(), 3.0, 1.0)
    confidence = confidence * np.log1p(interactions["timespent"].to_numpy(dtype=np.float32))
    matrix = sp.csr_matrix(
        (confidence, (user_codes, item_codes)),
        shape=(user_codes.max() + 1, len(item_ids)),
        dtype=np.float32,
    )
    model = AlternatingLeastSquares(
        factors=ALS_FACTORS,
        iterations=ALS_ITERATIONS,
        regularization=0.05,
        random_state=42,
    )
    model.fit(matrix, show_progress=False)
    return normalize_rows(model.item_factors.astype(np.float32))

als_embeddings = train_als_item_embeddings(interactions, item_ids)
als_embeddings.shape

(6240, 32)

## 5. Pair Feature Functions


In [32]:
item_feature_index = item_features.set_index("item_id")


def cosine_for_pairs(left_idx, right_idx, embeddings):
    return np.sum(embeddings[left_idx] * embeddings[right_idx], axis=1)


def add_pair_features(pairs, left_col, right_col):
    pairs = pairs.copy()
    pairs = pairs[pairs[left_col].isin(item_to_idx) & pairs[right_col].isin(item_to_idx)].copy()
    left_idx = pairs[left_col].map(item_to_idx).to_numpy()
    right_idx = pairs[right_col].map(item_to_idx).to_numpy()

    left_meta = item_feature_index.loc[pairs[left_col]].reset_index(drop=True)
    right_meta = item_feature_index.loc[pairs[right_col]].reset_index(drop=True)

    pairs["content_cos"] = cosine_for_pairs(left_idx, right_idx, content_embeddings)
    pairs["als_cos"] = cosine_for_pairs(left_idx, right_idx, als_embeddings)
    pairs["same_author"] = (left_meta["author_id"].to_numpy() == right_meta["author_id"].to_numpy()).astype(np.int8)
    pairs["duration_abs_diff"] = np.abs(
        left_meta["duration"].to_numpy(dtype=np.int16) - right_meta["duration"].to_numpy(dtype=np.int16)
    )
    pairs["log_shows_left"] = np.log1p(left_meta["shows"].to_numpy())
    pairs["log_shows_right"] = np.log1p(right_meta["shows"].to_numpy())
    pairs["like_rate_diff"] = left_meta["like_rate"].to_numpy() - right_meta["like_rate"].to_numpy()
    pairs["watch_time_diff"] = left_meta["mean_timespent"].to_numpy() - right_meta["mean_timespent"].to_numpy()
    return pairs

PAIR_FEATURES = [
    "content_cos",
    "als_cos",
    "same_author",
    "duration_abs_diff",
    "log_shows_left",
    "log_shows_right",
    "like_rate_diff",
    "watch_time_diff",
]


## 6. Join Features to Relevance Dataset


In [33]:
rel_train_features = add_pair_features(relevance_train, "leftId", "rightId")
rel_test_features = add_pair_features(relevance_test, "leftId", "rightId")

print(rel_train_features.shape, rel_test_features.shape)
rel_train_features.head()


(160000, 11) (40000, 11)


,leftId,rightId,verdict,content_cos,als_cos,same_author,duration_abs_diff,log_shows_left,log_shows_right,like_rate_diff,watch_time_diff
0,175970818,289549133,1,0.176743,-0.193143,0,54,6.810142,7.491645,-0.021242,32.507867
1,49413784,383102533,1,0.401285,0.330420,0,11,7.898782,7.636270,-0.010363,-1.013688
2,220082115,39180312,0,0.181743,0.447968,0,3,5.648974,5.723585,-0.012860,-11.671726
3,199030149,565137913,0,0.228407,0.186807,0,29,7.311218,2.944439,0.002674,10.369430
4,66676269,440528741,1,0.155552,0.165421,0,170,7.305188,6.308098,0.021713,-26.441968


## 7. Train CatBoost Relevance Formula


In [34]:
rel_model = CatBoostClassifier(
    iterations=300,
    depth=6,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    train_dir=str(SEMINAR_DIR / "catboost_info"),
    verbose=True,
)
rel_model.fit(
    rel_train_features[PAIR_FEATURES],
    rel_train_features["verdict"],
    eval_set=(rel_test_features[PAIR_FEATURES], rel_test_features["verdict"]),
    use_best_model=True,
)

rel_proba = rel_model.predict_proba(rel_test_features[PAIR_FEATURES])[:, 1]
print("ROC-AUC:", round(roc_auc_score(rel_test_features["verdict"], rel_proba), 4))
print("PR-AUC:", round(average_precision_score(rel_test_features["verdict"], rel_proba), 4))


0:	test: 0.8911928	best: 0.8911928 (0)	total: 15.8ms	remaining: 4.74s
1:	test: 0.9000867	best: 0.9000867 (1)	total: 26.5ms	remaining: 3.95s
2:	test: 0.9013937	best: 0.9013937 (2)	total: 38ms	remaining: 3.77s
3:	test: 0.9022022	best: 0.9022022 (3)	total: 49ms	remaining: 3.63s
4:	test: 0.9035480	best: 0.9035480 (4)	total: 59.6ms	remaining: 3.51s
5:	test: 0.9038595	best: 0.9038595 (5)	total: 69.8ms	remaining: 3.42s
6:	test: 0.9046116	best: 0.9046116 (6)	total: 80ms	remaining: 3.35s
7:	test: 0.9051843	best: 0.9051843 (7)	total: 89.7ms	remaining: 3.27s
8:	test: 0.9058304	best: 0.9058304 (8)	total: 100ms	remaining: 3.24s
9:	test: 0.9067107	best: 0.9067107 (9)	total: 110ms	remaining: 3.19s
10:	test: 0.9069398	best: 0.9069398 (10)	total: 121ms	remaining: 3.17s
11:	test: 0.9072842	best: 0.9072842 (11)	total: 131ms	remaining: 3.14s
12:	test: 0.9078012	best: 0.9078012 (12)	total: 141ms	remaining: 3.12s
13:	test: 0.9078381	best: 0.9078381 (13)	total: 152ms	remaining: 3.09s
14:	test: 0.9081519	best

## 8. Join Features to Attractiveness Triplets

For each triplet `(A, B, C)` we calculate pair features for `(A, B)` and `(A, C)`, then train on the difference `features(A, B) - features(A, C)`.


In [35]:
def add_triplet_features(triplets):
    winner_pairs = triplets[["anchorId", "winnerId"]].rename(columns={"anchorId": "leftId", "winnerId": "rightId"})
    loser_pairs = triplets[["anchorId", "loserId"]].rename(columns={"anchorId": "leftId", "loserId": "rightId"})

    winner = add_pair_features(winner_pairs, "leftId", "rightId").reset_index(drop=True)
    loser = add_pair_features(loser_pairs, "leftId", "rightId").reset_index(drop=True)
    result = triplets.reset_index(drop=True).copy()

    for feature in PAIR_FEATURES:
        result[f"winner_{feature}"] = winner[feature]
        result[f"loser_{feature}"] = loser[feature]
        result[f"delta_{feature}"] = result[f"winner_{feature}"] - result[f"loser_{feature}"]
    return result.dropna().reset_index(drop=True)

attr_train_features = add_triplet_features(attractiveness_train)
attr_test_features = add_triplet_features(attractiveness_test)
ATTR_FEATURES = [f"delta_{feature}" for feature in PAIR_FEATURES]

print(attr_train_features.shape, attr_test_features.shape)
attr_train_features.head()


(7969, 28) (1993, 28)


,anchorId,winnerId,loserId,target,winner_content_cos,loser_content_cos,delta_content_cos,winner_als_cos,loser_als_cos,delta_als_cos,winner_same_author,loser_same_author,delta_same_author,winner_duration_abs_diff,loser_duration_abs_diff,delta_duration_abs_diff,winner_log_shows_left,loser_log_shows_left,delta_log_shows_left,winner_log_shows_right,loser_log_shows_right,delta_log_shows_right,winner_like_rate_diff,loser_like_rate_diff,delta_like_rate_diff,winner_watch_time_diff,loser_watch_time_diff,delta_watch_time_diff
0,154887242,308804716,492866071,4.043051,0.157157,0.236110,-0.078953,-0.000691,-0.131971,0.131280,0,0,0,46,40,6,4.330733,4.330733,0.0,3.637586,7.116394,-3.478808,0.026667,0.012044,0.014622,-11.082523,3.605448,-14.687971
1,76978620,390986725,327484273,4.852030,0.324760,0.200810,0.123950,0.066649,0.619387,-0.552737,0,0,0,10,65,-55,8.094989,8.094989,0.0,7.117206,8.414939,-1.297733,-0.048085,-0.019550,-0.028535,-35.931348,7.890148,-43.821497
2,586164105,329362081,517932985,4.852030,0.023491,0.118905,-0.095414,0.516303,0.030132,0.486171,0,0,0,24,55,-31,8.091627,8.091627,0.0,8.944289,6.985642,1.958648,0.006565,0.003645,0.002921,-122.142062,-1.194341,-120.947722
3,424834082,278659326,45421954,4.852030,0.374826,0.285130,0.089696,0.021900,0.190351,-0.168450,0,0,0,111,71,40,8.500454,8.500454,0.0,5.459586,6.269096,-0.809511,0.024002,0.028275,-0.004274,-18.860376,-11.822612,-7.037764
4,408285968,316241258,284345673,4.852030,0.076757,0.221066,-0.144309,0.088118,0.350679,-0.262561,0,0,0,134,34,100,8.367300,8.367300,0.0,5.886104,7.106606,-1.220502,0.013947,0.015092,-0.001145,-27.905671,-6.865677,-21.039994


## 9. Train CatBoost Attractiveness Formula


In [ ]:
attr_model = CatBoostRegressor(
    iterations=300,
    depth=6,
    learning_rate=0.05,
    loss_function="RMSE",
    eval_metric="MAE",
    random_seed=42,
    train_dir=str(SEMINAR_DIR / "catboost_info"),
    verbose=True,
)
attr_model.fit(
    attr_train_features[ATTR_FEATURES],
    attr_train_features["target"],
    eval_set=(attr_test_features[ATTR_FEATURES], attr_test_features["target"]),
    use_best_model=True,
)

attr_pred = attr_model.predict(attr_test_features[ATTR_FEATURES])
print("MAE:", round(mean_absolute_error(attr_test_features["target"], attr_pred), 4))
print("Correct winner share:", round(float((attr_pred > 0).mean()), 4))

0:	learn: 0.2457753	test: 0.2406916	best: 0.2406916 (0)	total: 2.43ms	remaining: 727ms
1:	learn: 0.2446620	test: 0.2397468	best: 0.2397468 (1)	total: 5.04ms	remaining: 751ms
2:	learn: 0.2435333	test: 0.2388433	best: 0.2388433 (2)	total: 6.3ms	remaining: 624ms
3:	learn: 0.2425239	test: 0.2380227	best: 0.2380227 (3)	total: 7.42ms	remaining: 549ms
4:	learn: 0.2415802	test: 0.2371928	best: 0.2371928 (4)	total: 8.85ms	remaining: 522ms
5:	learn: 0.2406530	test: 0.2364744	best: 0.2364744 (5)	total: 10.3ms	remaining: 506ms
6:	learn: 0.2398145	test: 0.2357515	best: 0.2357515 (6)	total: 11.7ms	remaining: 492ms
7:	learn: 0.2390565	test: 0.2351954	best: 0.2351954 (7)	total: 12.9ms	remaining: 471ms
8:	learn: 0.2383144	test: 0.2346046	best: 0.2346046 (8)	total: 14ms	remaining: 454ms
9:	learn: 0.2375662	test: 0.2340568	best: 0.2340568 (9)	total: 15.2ms	remaining: 441ms
10:	learn: 0.2368899	test: 0.2335195	best: 0.2335195 (10)	total: 16.3ms	remaining: 427ms
11:	learn: 0.2362752	test: 0.2330537	best: 0

## 10. Build kNN Indices

We use two embedding spaces: content and ALS. In a real service, each extra embedding source is just another retrieval channel and another group of pair features.


In [37]:
content_index = NearestNeighbors(n_neighbors=51, metric="cosine")
content_index.fit(content_embeddings)

als_index = NearestNeighbors(n_neighbors=51, metric="cosine")
als_index.fit(als_embeddings)


,n_neighbors,51
,radius,1.0
,algorithm,'auto'
,leaf_size,30
,metric,'cosine'
,p,2
,metric_params,None
,n_jobs,None


## 11. Full Item2Item Service Demo


In [38]:
def retrieve_from_index(source_item_id, index, embeddings, k=50):
    source_idx = item_to_idx[int(source_item_id)]
    _, indices = index.kneighbors(embeddings[source_idx : source_idx + 1], n_neighbors=k + 1)
    candidate_ids = item_ids[indices[0]]
    return [int(item_id) for item_id in candidate_ids if int(item_id) != int(source_item_id)]


def generate_candidates(user_id, k_per_index=10):
    positives = user_positives[user_positives["user_id"] == user_id]["item_id"].drop_duplicates().tolist()
    known = set(map(int, positives))
    rows = []
    for source_item_id in positives:
        if int(source_item_id) not in item_to_idx:
            continue
        for source_name, index, embeddings in [
            ("content", content_index, content_embeddings),
            ("als", als_index, als_embeddings),
        ]:
            for candidate_item_id in retrieve_from_index(source_item_id, index, embeddings, k=k_per_index):
                if candidate_item_id in known:
                    continue
                rows.append((int(source_item_id), candidate_item_id, source_name))
    candidates = pd.DataFrame(rows, columns=["source_item_id", "candidate_item_id", "retrieval_source"])
    if candidates.empty:
        return candidates
    return candidates.drop_duplicates(["source_item_id", "candidate_item_id"]).reset_index(drop=True)


def relevance_filter(candidates, threshold=0.7):
    scored = add_pair_features(
        candidates.rename(columns={"source_item_id": "leftId", "candidate_item_id": "rightId"}),
        "leftId",
        "rightId",
    ).rename(columns={"leftId": "source_item_id", "rightId": "candidate_item_id"})
    scored["relevance_score"] = rel_model.predict_proba(scored[PAIR_FEATURES])[:, 1]
    return scored[scored["relevance_score"] >= threshold].copy()


In [39]:
def attractiveness_tournament(scored_candidates, top_n=20, max_candidates_per_source=20):
    if scored_candidates.empty:
        return scored_candidates

    ranked_groups = []
    for source_item_id, group in scored_candidates.groupby("source_item_id"):
        group = (
            group.sort_values("relevance_score", ascending=False)
            .head(max_candidates_per_source)
            .copy()
            .reset_index(drop=True)
        )
        scores = np.zeros(len(group), dtype=np.float32)
        base_features = group[PAIR_FEATURES].copy()

        for i, j in combinations(range(len(group)), 2):
            delta = {f"delta_{feature}": base_features.loc[i, feature] - base_features.loc[j, feature] for feature in PAIR_FEATURES}
            pred = float(attr_model.predict(pd.DataFrame([delta]))[0])
            scores[i] += pred
            scores[j] -= pred

        group["attractiveness_score"] = scores
        ranked_groups.append(group)

    ranked = pd.concat(ranked_groups, ignore_index=True)
    ranked["final_score"] = ranked["relevance_score"] + 0.1 * ranked["attractiveness_score"]
    return ranked.sort_values("final_score", ascending=False).head(top_n).reset_index(drop=True)


In [40]:
user_activity = user_positives.groupby("user_id").size().sort_values(ascending=False)
demo_user_id = int(user_activity.index[0])
print("demo_user_id:", demo_user_id)
print("positive source items:", user_activity.iloc[0])

raw_candidates = generate_candidates(demo_user_id, k_per_index=10)
print("raw candidates:", len(raw_candidates))

relevant_candidates = relevance_filter(raw_candidates, threshold=0.5)
print("after relevance filter:", len(relevant_candidates))

recommendations = attractiveness_tournament(relevant_candidates, top_n=20, max_candidates_per_source=20)
recommendations[[
    "source_item_id",
    "candidate_item_id",
    "retrieval_source",
    "relevance_score",
    "attractiveness_score",
    "final_score",
]].head(20)


demo_user_id: 43204146
positive source items: 424
raw candidates: 7041
after relevance filter: 4671


,source_item_id,candidate_item_id,retrieval_source,relevance_score,attractiveness_score,final_score
0,135145269,487546840,als,0.984119,81.227409,9.106860
1,237704176,480947473,als,0.981753,76.668289,8.648582
2,351875474,395330063,als,0.960987,76.669914,8.627978
3,43860813,351643887,als,0.967948,76.497330,8.617681
4,207007886,42325426,als,0.972357,76.124535,8.584811
5,133535057,441157292,als,0.883582,76.595467,8.543129
6,67299775,483403128,als,0.993244,72.557411,8.248985
7,49682960,494634113,als,0.957974,72.393326,8.197307
8,570731678,303076525,als,0.989991,72.013565,8.191347
9,184249294,298859480,als,0.966824,72.223213,8.189146


## 12. Discussion

- Item2Item service is candidate generation, not final ranking.
- Relevance removes bad pairs; attractiveness orders the good ones.
- Dataset scripts store labels only; feature joins live in the notebook so the training-serving contract is visible.
- More embeddings can be added as extra kNN channels and extra pair similarity features.
